# Phase 6 — Sentence-Level Logit Confidence Baseline (B1)

> **IMPORTANT**: This notebook demonstrates a **sentence-level logit-confidence uncertainty baseline**.
> This is **NOT** sentence-level Semantic Energy.
> **No semantic clustering is performed** — this is a lightweight single-pass approximation
> of sentence-level uncertainty from logit statistics alone.
> All outputs from this notebook are labeled `method: "sentence_logit_baseline"` throughout.

---

**What this does**:
1. Split the generated answer into sentences using `pysbd`
2. Map each sentence back to its token span using the tokenizer
3. Compute per-sentence logit features: mean chosen-token logit + mean top1-top2 margin
4. Assign a per-sentence uncertainty score based on these logit statistics
5. Return the sentence with the **lowest** logit confidence as the highest-risk sentence

**Why this is useful**: Fast per-sentence risk attribution without any clustering or multi-sample generation.

**Why this is NOT Semantic Energy**: It does not generate multiple samples, does not cluster semantically
equivalent responses, and does not compute any flow-based energy score.

## Section 1 — Setup

In [ ]:
import sys
import os
import re
import numpy as np
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT    = os.path.abspath(os.path.join(os.getcwd(), '..'))
BACKEND_PATH = os.path.join(REPO_ROOT, 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
try:
    import pysbd
    print(f"pysbd {pysbd.__version__} available")
except ImportError:
    print("pysbd not installed. Run: pip install pysbd")
    raise

## Section 2 — Sentence Segmentation

We use `pysbd` for robust sentence boundary detection, which handles abbreviations, decimals, and other edge cases that naive splitting on `.` fails on.

In [ ]:
import pysbd

sentence_segmenter = pysbd.Segmenter(language='en', clean=False)

def split_into_sentences(text):
    """Split text into sentences using pysbd. Returns list of sentence strings."""
    sentences = sentence_segmenter.segment(text)
    return [s.strip() for s in sentences if s.strip()]

# Test
test_text = "Paris is the capital of France. It has a population of about 2.1 million. The Eiffel Tower was built in 1889."
sents = split_into_sentences(test_text)
print(f"Sentence split: {len(sents)} sentences")
for i, s in enumerate(sents):
    print(f"  [{i}] {s}")

## Section 3 — Token-to-Sentence Alignment

We map each generated token back to its sentence by character-span alignment.

In [ ]:
def align_tokens_to_sentences(tokenizer, answer_text, token_ids, sentences):
    """
    Map each token in token_ids to the sentence index it belongs to.
    
    Returns:
        token_sentence_idx: list[int] — sentence index for each token (len = len(token_ids))
    """
    # Build character-level offset for each token using char_to_token mapping
    # Tokenize the full answer text with return_offsets_mapping
    encoding = tokenizer(answer_text, return_offsets_mapping=True, add_special_tokens=False)
    offset_mapping = encoding['offset_mapping']  # list of (char_start, char_end) per token
    
    # Build sentence character spans
    sent_spans = []
    pos = 0
    for sent in sentences:
        start = answer_text.find(sent, pos)
        if start == -1:
            start = pos
        end = start + len(sent)
        sent_spans.append((start, end))
        pos = end
    
    # For each token in token_ids (which may be a subset of encoding tokens),
    # find which sentence it belongs to based on char offset
    token_sentence_idx = []
    enc_ids = encoding['input_ids']
    
    # Align token_ids to enc_ids positions (they should match; drop extra EOS if any)
    n = min(len(token_ids), len(offset_mapping))
    for tok_pos in range(n):
        char_start, char_end = offset_mapping[tok_pos]
        char_mid = (char_start + char_end) // 2
        sent_idx = len(sentences) - 1  # default to last sentence
        for si, (ss, se) in enumerate(sent_spans):
            if ss <= char_mid < se:
                sent_idx = si
                break
        token_sentence_idx.append(sent_idx)
    
    return token_sentence_idx

print("align_tokens_to_sentences defined.")

## Section 4 — Per-Sentence Logit Scoring

> **B1 baseline method — not Semantic Energy**

For each sentence, we compute:
- `mean_chosen_logit`: mean logit of the chosen token across tokens in this sentence
- `mean_logit_margin`: mean top1-top2 logit margin across tokens in this sentence
- `sentence_confidence`: aggregate confidence score for the sentence

In [ ]:
def score_sentences_from_logits(sentences, token_sentence_idx, logits, top2_logits):
    """
    Compute per-sentence logit confidence scores.
    
    This is the B1 sentence-level logit baseline — NOT Semantic Energy.
    No clustering, no multi-sample generation.
    
    Args:
        sentences: list of sentence strings
        token_sentence_idx: list[int] — sentence index per token
        logits: list[float] — chosen-token logit per step
        top2_logits: list[(float, float)] — (top1, top2) logit per step, or None
    
    Returns:
        list of dicts, one per sentence, with confidence metrics
    """
    n_sents = len(sentences)
    n_tokens = min(len(token_sentence_idx), len(logits))
    
    # Group tokens by sentence
    sent_logits   = [[] for _ in range(n_sents)]
    sent_margins  = [[] for _ in range(n_sents)]
    
    for i in range(n_tokens):
        si = token_sentence_idx[i]
        if 0 <= si < n_sents:
            sent_logits[si].append(logits[i])
            if top2_logits is not None and i < len(top2_logits):
                t1, t2 = top2_logits[i]
                margin = t1 - t2
                if np.isfinite(margin):
                    sent_margins[si].append(margin)
    
    # Compute valid sentence logits for min-max normalisation
    all_mean_logits = []
    for sl in sent_logits:
        if sl:
            all_mean_logits.append(float(np.mean(sl)))
    
    logit_min = min(all_mean_logits) if all_mean_logits else 0.0
    logit_max = max(all_mean_logits) if all_mean_logits else 1.0
    logit_range = logit_max - logit_min if logit_max > logit_min else 1.0

    results = []
    for si, sent in enumerate(sentences):
        sl = sent_logits[si]
        sm = sent_margins[si]
        
        if len(sl) == 0:
            results.append({
                'sentence': sent,
                'num_tokens': 0,
                'mean_chosen_logit': None,
                'mean_logit_margin': None,
                'sentence_confidence': None,
                'method': 'sentence_logit_baseline',
            })
            continue
        
        mean_logit  = float(np.mean(sl))
        mean_margin = float(np.mean(sm)) if sm else None
        
        # Min-max normalise across sentences in THIS response so scores spread [0, 1].
        # Higher logit = higher confidence.
        sentence_confidence = (mean_logit - logit_min) / logit_range
        
        results.append({
            'sentence': sent,
            'num_tokens': len(sl),
            'mean_chosen_logit': mean_logit,
            'mean_logit_margin': mean_margin,
            'sentence_confidence': sentence_confidence,
            'method': 'sentence_logit_baseline',
        })
    
    return results

print("score_sentences_from_logits defined.")

## Section 5 — End-to-End Demo

Run a full example through the B1 baseline.

In [ ]:
from engine import SemanticEngine

print("Loading model for demo...")
engine = SemanticEngine(model_id='meta-llama/Llama-3.1-8B-Instruct', use_8bit=True)
print("Model ready.")

In [ ]:
question = "Tell me about the history of the Eiffel Tower."

print(f"Question: {question}")
print()

# Generate ONE response (B1 baseline — single generation)
gen_data = engine.generate_responses(question, num_samples=1)
answer   = gen_data[0]['answer']
logits   = gen_data[0]['logits']
token_ids = gen_data[0]['token_ids']
top2_logits = gen_data[0].get('top2_logits', None)

print(f"Answer ({len(token_ids)} tokens):")
print(answer)
print()

In [ ]:
# Sentence splitting
sentences = split_into_sentences(answer)
print(f"Sentences: {len(sentences)}")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s[:80]}{'...' if len(s) > 80 else ''}")

In [ ]:
# Token-to-sentence alignment
token_sent_idx = align_tokens_to_sentences(engine.tokenizer, answer, token_ids, sentences)
print(f"Token assignments: {len(token_sent_idx)} tokens mapped to {len(sentences)} sentences")
for si in range(len(sentences)):
    count = sum(1 for x in token_sent_idx if x == si)
    print(f"  Sentence {si}: {count} tokens")

In [ ]:
# Per-sentence logit scoring
sentence_scores = score_sentences_from_logits(sentences, token_sent_idx, logits, top2_logits)

print()
print("=" * 70)
print("B1 SENTENCE-LEVEL LOGIT BASELINE — method: sentence_logit_baseline")
print("(NOT Semantic Energy — no clustering, no multi-sample generation)")
print("=" * 70)
print()

for r in sentence_scores:
    if r['sentence_confidence'] is None:
        continue
    risk_flag = " ← LOWEST CONFIDENCE" if r['sentence_confidence'] == min(
        x['sentence_confidence'] for x in sentence_scores if x['sentence_confidence'] is not None
    ) else ""
    margin_str = f"{r['mean_logit_margin']:+.2f}" if r['mean_logit_margin'] is not None else 'N/A'
    print(f"Confidence: {r['sentence_confidence']:.3f}  | Logit: {r['mean_chosen_logit']:+.2f}  | Margin: {margin_str}{risk_flag}")
    print(f"  \"{r['sentence'][:90]}{'...' if len(r['sentence']) > 90 else ''}\"")
    print()

## Section 6 — API Integration Note

The B1 sentence baseline is available via the `/chat` endpoint — the sentence-level scores
can be added as an optional field in the response. A dedicated `/score_sentence_baseline`
endpoint returning per-sentence risk scores can be added to `backend/app.py` if needed.

---

> **Reminder**: This is the **B1 logit baseline**.
> The proper sentence-level Semantic Energy approach (B2) requires:
> - Claim decomposition of each sentence
> - N diverse generations per claim
> - Claim-level semantic clustering
> - Claim-level energy/entropy computation
>
> B2 is a separate, future phase. B1 is a fast approximation baseline only.